In [1]:
# Cell 1: Import mô-đun và Tải dữ liệu thô
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(".."))
from utils.data_fetcher import StockDataFetcher
from utils.data_preprocessor import DataPreprocessor

# 1. Khởi tạo Fetcher & Preprocessor
fetcher = StockDataFetcher()
preprocessor = DataPreprocessor()

# 2. Đọc dữ liệu thô AAPL đã tải từ Notebook 02
df_raw = fetcher.load_from_sqlite("AAPL")
print("=== DỮ LIỆU BAN ĐẦU ===")
print(df_raw.info())
print(f"Số lượng missing values ban đầu:\n{df_raw.isnull().sum()}")

# Cell 2: Làm sạch & Điền khuyết an toàn
df_clean = preprocessor.clean_time_series(df_raw, freq="D")

# Cell 3: Xử lý Outliers bằng IQR Clipping
cols_to_check = ["Open", "High", "Low", "Close", "Volume"]
df_no_outliers = preprocessor.handle_outliers_iqr(df_clean, columns=cols_to_check, factor=1.5)

# Cell 4: Thử nghiệm Resample sang khung Tuần (Weekly)
df_weekly = preprocessor.resample_ohlcv(df_no_outliers, rule="W")
print("\n=== DỮ LIỆU KHUNG TUẦN (WEEKLY OHLCV) ===")
print(df_weekly.head())

# Cell 5: Thực hiện Scaling (Robust Scaler)
df_scaled = preprocessor.fit_transform_scale(df_no_outliers, columns=["Close", "Volume"], method="robust")
print("\n=== DỮ LIỆU SAU KHI ROBUST SCALING ===")
print(df_scaled[["Close", "Volume"]].head())

# Cell 6: Lưu dữ liệu đã tiền xử lý vào data/processed/
os.makedirs("../data/processed", exist_ok=True)
processed_path = "../data/processed/AAPL_cleaned.parquet"
df_no_outliers.to_parquet(processed_path, engine="pyarrow")
print(f"\n[Thành công] Đã lưu dữ liệu sạch vào: {processed_path}")

2026-07-29 22:33:41,350 [INFO] Đã làm sạch dữ liệu. Kích thước sau xử lý: (2191, 5)
2026-07-29 22:33:41,357 [INFO] Cột 'Open': Phát hiện & Clip 0 giá trị ngoại lệ.
2026-07-29 22:33:41,365 [INFO] Cột 'High': Phát hiện & Clip 0 giá trị ngoại lệ.
2026-07-29 22:33:41,374 [INFO] Cột 'Low': Phát hiện & Clip 0 giá trị ngoại lệ.
2026-07-29 22:33:41,380 [INFO] Cột 'Close': Phát hiện & Clip 0 giá trị ngoại lệ.
2026-07-29 22:33:41,394 [INFO] Cột 'Volume': Phát hiện & Clip 41 giá trị ngoại lệ.
2026-07-29 22:33:41,422 [INFO] Đã Resample dữ liệu sang khung 'W'. Kích thước mới: (314, 5)
2026-07-29 22:33:41,452 [INFO] Đã Scale các cột ['Close', 'Volume'] bằng phương pháp 'robust'.


=== DỮ LIỆU BAN ĐẦU ===
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1508 entries, 2020-01-02 to 2025-12-31
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Open    1508 non-null   float64
 1   High    1508 non-null   float64
 2   Low     1508 non-null   float64
 3   Close   1508 non-null   float64
 4   Volume  1508 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 70.7 KB
None
Số lượng missing values ban đầu:
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

=== DỮ LIỆU KHUNG TUẦN (WEEKLY OHLCV) ===
                 Open       High        Low      Close       Volume
Date                                                               
2020-01-05  71.344054  72.394086  71.091184  71.630638  281803200.0
2020-01-12  70.754021  75.300919  70.503554  74.737366  670091600.0
2020-01-19  75.052871  76.762779  74.549522  76.760376  652055600.0
2020-01-26  76.167924  77.868168  75.862070  76.